# 01. UI video sampling과 temporal masking

목표: 길이가 다른 UI video에서 16개 frame을 균등 sampling하고, 2-frame tubelet 단위의 spatial·temporal mask를 만들어 봅니다. 표준 라이브러리만 사용하며 원 논문의 학습 코드를 완전히 재현하지 않는 toy 실습입니다.

In [ ]:
def evenly_spaced_indices(total_frames, sample_count=16):
    if total_frames < sample_count:
        raise ValueError('UI-JEPA 전처리와 같이 16 frame 미만 video는 제외합니다.')
    return [round(i * (total_frames - 1) / (sample_count - 1)) for i in range(sample_count)]

for length in (16, 53, 344, 749):
    indices = evenly_spaced_indices(length)
    print(f'{length:3d} frames -> {indices}')
    assert len(indices) == 16 and indices[0] == 0 and indices[-1] == length - 1

고정 stride는 긴 video의 끝부분을 놓칠 수 있습니다. flexible stride는 task 전체를 포괄하지만 매우 짧은 클릭이나 전환이 sampling 사이에 사라질 수 있다는 trade-off가 있습니다.

In [ ]:
from random import Random

T, H, W = 8, 8, 8  # 16 frames / tubelet 2 = 8 hyper-frames
rng = Random(240904081)

def rectangle(top, left, height, width):
    return {(row, col) for row in range(top, top + height)
                       for col in range(left, left + width)}

# 설명을 위한 간단한 short/long spatial block입니다. 실제 sampler는 비율과 종횡비를 무작위화합니다.
spatial = rectangle(1, 1, 2, 3) | rectangle(3, 2, 4, 5)
spatial_volume = {(t, row, col) for t in range(T) for row, col in spatial}

# 논문에서 선택한 discrete temporal mask 6개를 hyper-frame 단위로 가정합니다.
masked_times = set(rng.sample(range(T), 6))
temporal_volume = {(t, row, col) for t in masked_times
                   for row in range(H) for col in range(W)}
combined = spatial_volume | temporal_volume
total = T * H * W
print('temporal mask hyper-frames:', sorted(masked_times))
print(f'combined mask ratio: {len(combined) / total:.1%}')
assert len(masked_times) == 6 and 0 < len(combined) <= total

In [ ]:
def show_timeline(masked):
    return ' '.join('FULL' if t in masked else 'PART' for t in range(T))

print(show_timeline(masked_times))
print('FULL은 화면 전체, PART는 공간 block 일부만 가린 hyper-frame입니다.')

## 생각해 볼 점

마지막 frame만 중요한 task에서는 6/8 hyper-frame을 가리는 것이 지나칠 수 있습니다. contiguous와 discrete mask를 app 전환 빈도, 평균 video 길이, zero-shot 성능별로 따로 검증해야 합니다.